In [ ]:
import pickle as pkl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az
import scipy.stats as stats

In [ ]:
#open the output file

import os

file_path = '/home/apoorva/Desktop/work/MZ_analysis/data/New output (pkl)/Steady State/SHM_T2.pkl'
if os.path.getsize(file_path) > 0:
    with open(file_path, 'rb') as f:
        data_neutral = pkl.load(f)
else:
    raise EOFError("File is empty")

In [ ]:
# open experimental data file 
df = pd.read_csv('/home/apoorva/Desktop/work/MZ_analysis/data/new_data_MZB.csv')
df = df.sort_values(by='Age at S1K')
print(df)


In [ ]:
#Add one column in df_Ki67 and df_Nfd for grouping the data based on age at bmt

df['agebin'] = pd.cut(df['Age at BMT'], bins=[0, 61, 81, 101], labels=['agebin1 = [41,61]', 'agebin2 = [62,81]', 'agebin3 = [82,101]'])
print(df)

In [ ]:

ts1= np.arange(42, 750, 1)
ts2= np.arange(66, 750, 1)
ts3= np.arange(88, 750, 1)

# Plotting
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 6)  # Create a GridSpec with 2 rows and 4 columns

# Create the subplots
ax1 = fig.add_subplot(gs[0, 0:3])  # First row, spanning columns 0 to 1
ax2 = fig.add_subplot(gs[0, 3:6])  # First row, spanning columns 2 to 3
# second row has 3 plots based on the age at BMT
ax_3 = fig.add_subplot(gs[1, 0:2])  # First row, spanning columns 0 to 1
ax_4 = fig.add_subplot(gs[1, 2:4])  # First row, spanning columns 2 to 3
ax_5 = fig.add_subplot(gs[1, 4:6])  # First row, spanning columns 2 to 3 

# Plot total MZ cell counts
sns.scatterplot(ax=ax1, data=df, x='Age at S1K', y='total_MZ', alpha=0.85, hue ='agebin', s=100, legend=False)
ax1.plot(ts1, np.mean(data_neutral.MZ_counts_pred1, axis=0))
ax1.plot(ts2, np.mean(data_neutral.MZ_counts_pred2, axis=0))
ax1.plot(ts3, np.mean(data_neutral.MZ_counts_pred3, axis=0), color='Green')
ax1.fill_between(ts1, np.quantile(data_neutral.MZ_counts_pred1, 0.025, axis=0), np.quantile(data_neutral.MZ_counts_pred1, 0.975, axis=0), alpha=0.2, color = 'steelblue')
ax1.fill_between(ts2, np.quantile(data_neutral.MZ_counts_pred2, 0.025, axis=0), np.quantile(data_neutral.MZ_counts_pred2, 0.975, axis=0), alpha=0.2, color= 'orange')
ax1.fill_between(ts3, np.quantile(data_neutral.MZ_counts_pred3, 0.025, axis=0), np.quantile(data_neutral.MZ_counts_pred3, 0.975, axis=0), alpha=0.2, color = 'green')
ax1.set_yscale('log')  # sets the y-axis to a logarithmic scale
ax1.set_ylim(1e5, 1e7)
ax1.set_xscale('log')
ax1.set_xlim(0, 800)
ax1.set_xticks([75, 150, 300, 600])
ax1.set_xticklabels([75, 150, 300, 600])
ax1.set_ylabel('Total MZ B cell numbers', fontsize=14)
ax1.set_xlabel('Host Age (days)', fontsize=14)
ax1.set_title('Variation in pool size across the lifetime', fontsize=14, fontweight='bold')
ax1.grid(True, which='both', ls="solid", linewidth=0.5, alpha=0.3)

# Plot normalized donor fraction

sns.scatterplot(ax=ax2, data=df, x='Age at S1K', y='Nfd_MZ', hue='agebin', alpha=0.85, s=100)
ax2.plot(ts1, np.mean(data_neutral.Nfd_pred1, axis=0))
ax2.plot(ts2, np.mean(data_neutral.Nfd_pred2, axis=0))
ax2.plot(ts3, np.mean(data_neutral.Nfd_pred3, axis=0))
ax2.fill_between(ts1, np.quantile(data_neutral.Nfd_pred1, 0.025, axis=0), np.quantile(data_neutral.Nfd_pred1, 0.975, axis=0), alpha=0.1, color= 'steelblue')
ax2.fill_between(ts2, np.quantile(data_neutral.Nfd_pred2, 0.025, axis=0), np.quantile(data_neutral.Nfd_pred2, 0.975, axis=0), alpha=0.1, color = 'orange')
ax2.fill_between(ts3, np.quantile(data_neutral.Nfd_pred3, 0.025, axis=0), np.quantile(data_neutral.Nfd_pred3, 0.975, axis=0), alpha=0.1, color = 'green')
# ax2.set_xscale('log')
ax2.set_xlim(0, 800)
ax2.set_xticks([50, 150,  300,  600])
ax2.set_xticklabels([50, 150,  300,  600])
# ax2.set_xscale('log')
#ax2.set_xticks(np.arange(0, 750, 100))
ax2.set_ylabel('')
ax2.set_yticks(np.arange(0, 1.25, 0.25))
ax2.set_xlabel('Host age (days)', fontsize=14)
ax2.set_title('Donor fraction in MZ B normalized to chimerism in Transitional_TAA4.1', fontsize=14, fontweight='bold')
ax2.axhline(y=1, color='darkred', linestyle='dashed', linewidth=0.85)
ax2.grid(True, which='both', ls="solid", linewidth=0.5, alpha=0.3)
ax2.legend(title='Age at BMT', fontsize=12)

#Construct a new dataframe for agebin1, agebin2 and agebin3
df_Ki67_bin1 = df[df['agebin'] == 'agebin1 = [41,61]']
df_Ki67_bin2 = df[df['agebin'] == 'agebin2 = [62,81]']
df_Ki67_bin3 = df[df['agebin'] == 'agebin3 = [82,101]']

# Plot Proportion of Ki67+ MZ cells for age bin1
sns.scatterplot(ax=ax_3, data=df_Ki67_bin1, x='Age at S1K', y='frac_Ki67+_MZ_host', alpha=0.85, s=100, color='blue')
sns.scatterplot(ax=ax_3, data=df_Ki67_bin1, x='Age at S1K', y='frac_Ki67+_MZ_donor', alpha=0.85, s=100, color='brown')
ax_3.plot(ts1, np.mean(data_neutral.donor_ki_pred1, axis=0), color= 'brown')
ax_3.fill_between(ts1, np.quantile(data_neutral.donor_ki_pred1, 0.025, axis=0), np.quantile(data_neutral.donor_ki_pred1, 0.975, axis=0), color= 'red', alpha=0.2)
ax_3.plot(ts1, np.mean(data_neutral.host_ki_pred1, axis=0), color = 'blue')
ax_3.fill_between(ts1, np.quantile(data_neutral.host_ki_pred1, 0.025, axis=0), np.quantile(data_neutral.host_ki_pred1, 0.975, axis=0), alpha=0.2)
ax_3.set_ylim(0, 1)
ax_3.set_yticks(np.arange(0, 1.25, 0.25))
ax_3.set_xscale('log')
ax_3.set_xlim(0, 800)
ax_3.set_xticks([50, 150, 300, 600])
ax_3.set_xticklabels([50, 150, 300, 600])
ax_3.set_ylabel('Proportion of Ki67+ MZ cells', fontsize=14)
ax_3.set_xlabel('Host Age (days)', fontsize=14)
# ax_3.set_title('Extent of cell division', fontsize=14, fontweight='bold')
ax_3.grid(True, ls="solid")


# Plot Proportion of Ki67+ MZ cells for age bin2
sns.scatterplot(ax=ax_4, data=df_Ki67_bin2, x='Age at S1K', y='frac_Ki67+_MZ_host', alpha=0.85, s=100, color='blue')
sns.scatterplot(ax=ax_4, data=df_Ki67_bin2, x='Age at S1K', y='frac_Ki67+_MZ_donor', alpha=0.85, s=100, color='brown')
ax_4.plot(ts2, np.mean(data_neutral.donor_ki_pred2, axis=0), color= 'brown')
ax_4.fill_between(ts2, np.quantile(data_neutral.donor_ki_pred2, 0.025, axis=0), np.quantile(data_neutral.donor_ki_pred2, 0.975, axis=0), color= 'red', alpha=0.2)
ax_4.plot(ts2, np.mean(data_neutral.host_ki_pred2, axis=0), color = 'blue')
ax_4.fill_between(ts2, np.quantile(data_neutral.host_ki_pred2, 0.025, axis=0), np.quantile(data_neutral.host_ki_pred2, 0.975, axis=0), alpha=0.2)
ax_4.set_ylim(0, 1)
ax_4.set_yticks(np.arange(0, 1.25, 0.25))
ax_4.set_xscale('log')
ax_4.set_xlim(0, 800)
ax_4.set_xticks([50, 150, 300, 600])
ax_4.set_xticklabels([50, 150, 300, 600])
ax_4.set_ylabel('')
ax_4.set_xlabel('Host Age (days)', fontsize=14)
ax_4.set_title('Extent of cell division', fontsize=14, fontweight='bold')
ax_4.grid(True, ls="solid")


sns.scatterplot(ax=ax_5, data=df_Ki67_bin3, x='Age at S1K', y='frac_Ki67+_MZ_host', alpha=0.85, s=100, color='blue')
sns.scatterplot(ax=ax_5, data=df_Ki67_bin3, x='Age at S1K', y='frac_Ki67+_MZ_donor', alpha=0.85, s=100, color='brown')
ax_5.plot(ts3, np.mean(data_neutral.donor_ki_pred3, axis=0), color= 'brown')
ax_5.fill_between(ts3, np.quantile(data_neutral.donor_ki_pred3, 0.025, axis=0), np.quantile(data_neutral.donor_ki_pred3, 0.975, axis=0), color= 'red', alpha=0.2)
ax_5.plot(ts3, np.mean(data_neutral.host_ki_pred3, axis=0), color = 'blue')
ax_5.fill_between(ts3, np.quantile(data_neutral.host_ki_pred3, 0.025, axis=0), np.quantile(data_neutral.host_ki_pred3, 0.975, axis=0), alpha=0.2)
ax_5.set_ylim(0, 1)
ax_5.set_yticks(np.arange(0, 1.25, 0.25))
ax_5.set_xscale('log')
ax_5.set_xlim(0, 800)
ax_5.set_xticks([50, 150, 300, 600])
ax_5.set_xticklabels([50, 150, 300, 600])
ax_5.set_ylabel('')
ax_5.set_xlabel('Host Age (days)', fontsize=14)
# ax_5.set_title('Extent of cell division', fontsize=14, fontweight='bold')
ax_5.grid(True, ls="solid")
ax_5.legend(labels=['Host', 'Donor'])

plt.tight_layout()

#Save the figure
# plt.savefig('/home/apoorva/Desktop/work/MZ_analysis/figures/new_output/SHM_T2MZP.png', dpi=300, bbox_inches='tight')


In [ ]:


fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot Residual Ki Donor
axes[1, 0].scatter(x=df['Age at S1K'], y=np.mean(data_neutral.residuals_Kid, axis=0), marker='o', color='b')
axes[1, 0].axhline(y=0, color='r', linestyle='--')
axes[1, 0].set_xlabel('Host age')
axes[1, 0].set_ylabel('Residual Ki Donor')
axes[1, 0].set_title('Residual Plot: Ki Donor')

# Plot Residual Ki Host
axes[1, 1].scatter(x=df['Age at S1K'], y=np.mean(data_neutral.residuals_Kih, axis=0), marker='o', color='b')
axes[1, 1].axhline(y=0, color='r', linestyle='--')
axes[1, 1].set_xlabel('Host age')
axes[1, 1].set_ylabel('Residual Ki Host')
axes[1, 1].set_title('Residual Plot: Ki Host')

# Plot Residual Nfd
axes[0, 1].scatter(x=df['Age at S1K'], y=np.mean(data_neutral.residuals_Nfd, axis=0), marker='o', color='b')
axes[0, 1].axhline(y=0, color='r', linestyle='--')
axes[0, 1].set_xlabel('Host age')
axes[0, 1].set_ylabel('Residual Nfd')
axes[0, 1].set_title('Residual Plot: Nfd')

# Plot Residual Mz Counts
axes[0, 0].scatter(x=df['Age at S1K'], y=np.mean(data_neutral.residuals_Mz, axis=0), marker='o', color='b')
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Host age')
axes[0, 0].set_ylabel('Residual Mz Counts')
axes[0, 0].set_title('Residual Plot: Mz Counts')

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
# Calculate elpdf for the model

idata = az.from_cmdstanpy(data_neutral, log_likelihood={"log_lik_Mz": "log_lik_Mz", "log_lik_Nfd": "log_lik_Nfd", "log_lik_Kid": "log_lik_Kid", "log_lik_Kih": "log_lik_Kih"})

# Compute Loo CV for models with several Likelihoods 
import xarray as xr

# Concatenate the log_likelihood arrays

a1 = np.array(idata.log_likelihood["log_lik_Mz"])
a2 = np.array(idata.log_likelihood["log_lik_Nfd"])
a3 = np.array(idata.log_likelihood["log_lik_Kid"])
a4 = np.array(idata.log_likelihood["log_lik_Kih"])

log_lik = np.concatenate((a1, a2, a3, a4), axis=2)

# Convert the concatenated array to an xarray DataArray
idata.log_likelihood["combined_log_lik"] = xr.DataArray(log_lik, dims=["chain", "draw", "log_likelihood_dim"], coords={"chain": np.arange(log_lik.shape[0]), "draw": np.arange(log_lik.shape[1]), "log_likelihood_dim": np.arange(log_lik.shape[2])})

# Compute loo_cv
loo = loo_result1_combined = az.loo(idata, var_name="combined_log_lik")
print(loo)

In [ ]:
# Posterior distributions of key parameters

az.plot_posterior(data_neutral, var_names=['rho', 'phi','kappa_0', 'y0'], kind='kde')

In [ ]:
# Calculate clonal lifespans

netloss = 1/ (data_neutral.delta - data_neutral.rho)
clonallife = np.log(2)*netloss
print(clonallife)
print(np.mean(clonallife))
sns.kdeplot(clonallife)


In [ ]:
# Define 2D array for ratio of influx produced by precursor cells to self-renewing cells in the MZ pool

influx = np.zeros((data_neutral.rho.shape[0], len(data_neutral.MZ_counts_pred1[0])))
for i in range(0, len(data_neutral.MZ_counts_pred1[0])):
	influx[:, i] = (data_neutral.phi * 1118208) / (data_neutral.rho * data_neutral.MZ_counts_pred1[:, i])
 

# Plot precursor influx with 95% CI

fig, axes = plt.subplots(1, 1, figsize=(8, 6))
axes.plot(ts1, np.mean(influx, axis=0))
axes.fill_between(ts1, np.quantile(influx, 0.025, axis=0), np.quantile(influx, 0.975, axis=0), alpha=0.3)
axes.set_ylim(0, 1.5)
axes.set_xlim(0, 730)
axes.set_xticks([50, 200, 350, 500])
axes.set_xticklabels([50, 200, 350, 500])
axes.set_ylabel('Precursor influx')
axes.set_xlabel('Host Age (days)')
axes.set_title('Ratio of influx produced by precursor cells to self-renewing cells in the MZ pool') 
axes.grid(True, which='both', ls="solid", linewidth=0.5, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Proportion of influx of TA44.1 precursor cells into the MZ pool
influx_p = np.zeros((data_neutral.rho.shape[0], len(data_neutral.MZ_counts_pred1[0])))
for i in range(0, len(data_neutral.MZ_counts_pred1[0])):
	influx_p[:, i] = (data_neutral.phi * 1118208) / data_neutral.MZ_counts_pred1[:, i]
 

# Plot precursor influx with 95% CI

fig, axes = plt.subplots(1, 1, figsize=(8, 6))
axes.plot(ts1, np.mean(influx_p, axis=0))
axes.fill_between(ts1, np.quantile(influx_p, 0.025, axis=0), np.quantile(influx_p, 0.975, axis=0), alpha=0.3)
axes.set_ylim(0, 0.02)
axes.set_yticks([0, 0.005, 0.01, 0.015, 0.02])
axes.set_yticklabels([0, 0.005, 0.01, 0.015, 0.02])
axes.set_xlim(0, 730)
axes.set_xticks([50, 200, 350, 500])
axes.set_xticklabels([50, 200, 350, 500])
axes.set_ylabel('Fraction of influx')
axes.set_xlabel('Host Age (days)')
axes.grid(True, which='both', ls="solid", linewidth=0.5, alpha=0.3)
axes.set_title('Proportion of influx of TA44.1 precursor cells into the MZ pool')
plt.tight_layout()
plt.show()


In [ ]:
# Daily influx of precursor cells

print(np.mean(data_neutral.phi*1308516))
print(np.std(data_neutral.phi*1308516))

# Calculate credible interval 

print(np.quantile(data_neutral.phi*1298783, 0.025))
print(np.quantile(data_neutral.phi*1298783, 0.975))